# FEATURE ENGINEERING

**Input:** `../data/processed/combined_jobs_cleaned.csv`
**Output:** `../data/features/combined_jobs_features.csv`

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 150, 'font.size': 11})

ROOT        = Path('..').resolve()
INPUT_PATH  = ROOT / 'data' / 'processed' / 'combined_jobs_cleaned.csv'
OUTPUT_PATH = ROOT / 'data' / 'features'  / 'combined_jobs_features.csv'
PLOTS_PATH  = ROOT / 'outputs' / 'plots'

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
PLOTS_PATH.mkdir(parents=True, exist_ok=True)

In [7]:
# LOAD
df = pd.read_csv(INPUT_PATH)
df['posted_date']    = pd.to_datetime(df['posted_date'],    errors='coerce', utc=True)
df['collected_date'] = pd.to_datetime(df['collected_date'], errors='coerce', utc=True)
print(f"Rows loaded: {len(df)}")

Rows loaded: 2722


In [8]:
# FEATURE 1 - MISSING COMPANY NAME
df['missing_company'] = (df['company_name'] == 'Unknown').astype(int)
print(f"\nF1 Missing company      : {df['missing_company'].sum()} ({df['missing_company'].mean()*100:.1f}%)")


F1 Missing company      : 20 (0.7%)


In [9]:
# FEATURE 2 - SALARY FLAGS
salary_95th = df['salary_avg'].quantile(0.95)
salary_5th  = df['salary_avg'].quantile(0.05)

df['salary_missing']   = df['salary_avg'].isna().astype(int)
df['high_salary_flag'] = (df['salary_avg'] > salary_95th).astype(int)
df['low_salary_flag']  = ((df['salary_avg'] < salary_5th) & df['salary_avg'].notna()).astype(int)

# Salary range spread - very wide range can be suspicious
df['salary_range'] = (df['salary_max'] - df['salary_min']).clip(lower=0)
range_95th = df['salary_range'].quantile(0.95)
df['wide_salary_range'] = (df['salary_range'] > range_95th).astype(int)

print(f"F2 High salary (>£{salary_95th:,.0f}) : {df['high_salary_flag'].sum()} ({df['high_salary_flag'].mean()*100:.1f}%)")
print(f"F2 Low salary (<£{salary_5th:,.0f})   : {df['low_salary_flag'].sum()} ({df['low_salary_flag'].mean()*100:.1f}%)")
print(f"F2 Missing salary          : {df['salary_missing'].sum()} ({df['salary_missing'].mean()*100:.1f}%)")
print(f"F2 Wide salary range       : {df['wide_salary_range'].sum()} ({df['wide_salary_range'].mean()*100:.1f}%)")

F2 High salary (>£119,522) : 123 (4.5%)
F2 Low salary (<£31,000)   : 122 (4.5%)
F2 Missing salary          : 276 (10.1%)
F2 Wide salary range       : 117 (4.3%)


In [10]:
# FEATURE 3 - DESCRIPTION FLAGS
if 'is_truncated' not in df.columns:
    df['is_truncated'] = df['job_description'].str.strip().str.endswith('\u2026').astype(int)
else:
    df['is_truncated'] = df['is_truncated'].astype(int)

if 'desc_word_count' not in df.columns:
    df['desc_word_count'] = df['job_description_clean'].str.split().str.len()

# Only flag as short if NOT truncated (truncation is an API limit, not a real short desc)
df['short_desc'] = np.where(
    df['is_truncated'] == 0,
    (df['desc_word_count'] < 50).astype(int), 0
)

# Generic / vague description - low lexical diversity
def lexical_diversity(text):
    """Ratio of unique words to total words. Low = repetitive/vague."""
    if pd.isna(text) or text == '':
        return np.nan
    words = str(text).lower().split()
    if len(words) == 0:
        return np.nan
    return round(len(set(words)) / len(words), 3)

df['lexical_diversity'] = df['job_description_clean'].apply(lexical_diversity)
ld_5th = df['lexical_diversity'].quantile(0.05)
df['low_lexical_diversity'] = (df['lexical_diversity'] < ld_5th).astype(int)

print(f"\nF3 Short description       : {df['short_desc'].sum()} ({df['short_desc'].mean()*100:.1f}%)")
print(f"F3 Low lexical diversity   : {df['low_lexical_diversity'].sum()} ({df['low_lexical_diversity'].mean()*100:.1f}%)")


F3 Short description       : 7 (0.3%)
F3 Low lexical diversity   : 137 (5.0%)


In [11]:
# FEATURE 4 - SUSPICIOUS KEYWORDS
# Grounded in: Baraneetharan (2022), Ch'ng & Yi (2025), Pillai (2023)

SUSPICIOUS_KEYWORDS = [
    # Unrealistic earnings
    'easy money', 'quick cash', 'earn fast', 'unlimited earnings',
    'earn from home', 'passive income', 'make money online',
    # Urgency & vagueness
    'urgent hiring', 'immediate joining', 'no experience needed',
    'no experience required', 'no qualification needed',
    # Work from home scam patterns
    'work from home and earn', 'online earning', 'data entry from home',
    # Payment / info harvesting
    'registration fee', 'pay to apply', 'training fee', 'security deposit',
    # Contact via informal channels
    'send your cv to whatsapp', 'whatsapp your cv', 'apply via telegram',
    'dm for details', 'message us on whatsapp',
    # Classic scam phrases
    'be your own boss', 'financial freedom', 'limited slots',
    'join our team today and earn', 'guaranteed income',
]

def flag_keywords(text):
    if pd.isna(text):
        return 0
    tl = str(text).lower()
    return int(any(kw in tl for kw in SUSPICIOUS_KEYWORDS))

def get_keywords(text):
    if pd.isna(text):
        return ''
    tl = str(text).lower()
    return ', '.join(kw for kw in SUSPICIOUS_KEYWORDS if kw in tl)

df['suspicious_keywords'] = df['job_description'].apply(flag_keywords)
df['matched_keywords']    = df['job_description'].apply(get_keywords)

print(f"\nF4 Suspicious keywords     : {df['suspicious_keywords'].sum()} ({df['suspicious_keywords'].mean()*100:.1f}%)")


F4 Suspicious keywords     : 2 (0.1%)


In [12]:
# FEATURE 5 - LOCATION FLAGS
VAGUE_LOCATIONS = ['unknown', 'uk', 'united kingdom', 'remote', 'anywhere',
                   'us', 'usa', 'united states', 'worldwide', 'global']

df['vague_location'] = df['location'].apply(
    lambda x: 1 if str(x).strip().lower() in VAGUE_LOCATIONS else 0
)
print(f"\nF5 Vague location          : {df['vague_location'].sum()} ({df['vague_location'].mean()*100:.1f}%)")


F5 Vague location          : 378 (13.9%)


In [13]:
# FEATURE 6 - CONTRACT MISSING
df['contract_missing'] = (
    (df['contract_type'] == 'Unknown') & (df['contract_time'] == 'Unknown')
).astype(int)
print(f"\nF6 Contract info missing   : {df['contract_missing'].sum()} ({df['contract_missing'].mean()*100:.1f}%)")


F6 Contract info missing   : 757 (27.8%)


In [14]:
# FEATURE 7 - TITLE FLAGS
if 'title_word_count' not in df.columns:
    df['title_word_count'] = df['job_title'].str.split().str.len()

# Very short title (1 word) or very long (>10 words) can be suspicious
df['suspicious_title'] = (
    (df['title_word_count'] <= 1) | (df['title_word_count'] > 10)
).astype(int)

# Title contains ALL CAPS words (urgency signal)
df['title_has_caps'] = df['job_title'].apply(
    lambda x: int(bool(re.search(r'\b[A-Z]{3,}\b', str(x))))
)

print(f"\nF7 Suspicious title length : {df['suspicious_title'].sum()} ({df['suspicious_title'].mean()*100:.1f}%)")
print(f"F7 Title has ALL CAPS      : {df['title_has_caps'].sum()} ({df['title_has_caps'].mean()*100:.1f}%)")


F7 Suspicious title length : 37 (1.4%)
F7 Title has ALL CAPS      : 248 (9.1%)


In [15]:
# FEATURE 8 - CONTACT INFO IN DESCRIPTION
if 'contact_in_desc' not in df.columns:
    df['contact_in_desc'] = df['job_description_clean'].str.contains(
        r'\b(whatsapp|telegram|gmail|yahoo|hotmail|call us|text us)\b', regex=True
    ).astype(int)
print(f"\nF8 Contact info in desc    : {df['contact_in_desc'].sum()} ({df['contact_in_desc'].mean()*100:.1f}%)")


F8 Contact info in desc    : 8 (0.3%)


In [16]:
# COMPOSITE FRAUD RISK SCORE (weighted)
print("COMPOSITE FRAUD RISK SCORE")

# Weights grounded in literature importance
WEIGHTS = {
    'missing_company'      : 3,   # very strong signal (Ch'ng & Yi, 2025)
    'suspicious_keywords'  : 3,   # very strong signal (Pillai, 2023)
    'contact_in_desc'      : 3,   # very strong signal
    'high_salary_flag'     : 2,   # strong signal
    'short_desc'           : 2,   # strong for non-truncated
    'low_lexical_diversity': 2,   # strong signal
    'salary_missing'       : 1,
    'vague_location'       : 1,
    'contract_missing'     : 1,
    'suspicious_title'     : 1,
    'title_has_caps'       : 1,
    'wide_salary_range'    : 1,
}

MAX_SCORE = sum(WEIGHTS.values())

df['fraud_risk_score'] = sum(df[f] * w for f, w in WEIGHTS.items())
df['fraud_risk_pct']   = (df['fraud_risk_score'] / MAX_SCORE * 100).round(1)

def assign_tier(pct):
    if pct >= 40:   return 'High'
    elif pct >= 15: return 'Medium'
    else:           return 'Low'

df['risk_tier'] = df['fraud_risk_pct'].apply(assign_tier)

print(f"Max possible score : {MAX_SCORE}")
print(f"\nRisk tier distribution:")
print(df['risk_tier'].value_counts().to_string())
print(f"\nFraud risk score stats:")
print(df['fraud_risk_pct'].describe().round(1).to_string())

COMPOSITE FRAUD RISK SCORE
Max possible score : 21

Risk tier distribution:
risk_tier
Low       2555
Medium     166
High         1

Fraud risk score stats:
count    2722.0
mean        4.3
std         5.8
min         0.0
25%         0.0
50%         4.8
75%         4.8
max        47.6


In [17]:
# TRANSPARENCY EXPLANATION
def generate_explanation(row):
    reasons = []
    if row['missing_company']       == 1: reasons.append("Company information is missing")
    if row['high_salary_flag']      == 1: reasons.append(f"Salary unusually high (>£{salary_95th:,.0f})")
    if row['salary_missing']        == 1: reasons.append("No salary information provided")
    if row['short_desc']            == 1: reasons.append("Job description is very short")
    if row['low_lexical_diversity'] == 1: reasons.append("Description uses repetitive or vague language")
    if row['suspicious_keywords']   == 1: reasons.append(f"Suspicious language: '{row['matched_keywords']}'")
    if row['contact_in_desc']       == 1: reasons.append("Informal contact channel mentioned (WhatsApp/Telegram/Gmail)")
    if row['vague_location']        == 1: reasons.append("Location is vague or unspecified")
    if row['contract_missing']      == 1: reasons.append("No contract type or working hours provided")
    if row['suspicious_title']      == 1: reasons.append("Job title length is unusual")
    if row['title_has_caps']        == 1: reasons.append("Job title contains all-caps words")
    if row['wide_salary_range']     == 1: reasons.append("Salary range is unusually wide")
    return " | ".join(reasons) if reasons else "No suspicious indicators detected."

df['flag_explanation'] = df.apply(generate_explanation, axis=1)

print("\n--- Sample HIGH RISK jobs ---")
high = df[df['risk_tier'] == 'High'][
    ['job_title', 'company_name', 'fraud_risk_pct', 'flag_explanation']
].head(5)
print(high.to_string(index=False))

df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved → {OUTPUT_PATH}")


--- Sample HIGH RISK jobs ---
                                                      job_title company_name  fraud_risk_pct                                                                                                                                                                                                                                flag_explanation
Hi, I need a simple one-page website for an adult BDSM service.      Unknown            47.6 Company information is missing | Informal contact channel mentioned (WhatsApp/Telegram/Gmail) | Location is vague or unspecified | No contract type or working hours provided | Job title length is unusual | Job title contains all-caps words

Saved → C:\Users\Nethma\Desktop\Assigment\18k research\webscrape\AI_trust\2_development\data\features\combined_jobs_features.csv


In [18]:
# PLOTS
print("\nGenerating plots...")

ALL_FEATURES = {
    'missing_company'      : 'Missing Company',
    'high_salary_flag'     : 'High Salary',
    'salary_missing'       : 'Salary Missing',
    'short_desc'           : 'Short Description',
    'low_lexical_diversity': 'Low Lexical Diversity',
    'suspicious_keywords'  : 'Suspicious Keywords',
    'contact_in_desc'      : 'Informal Contact in Desc',
    'vague_location'       : 'Vague Location',
    'contract_missing'     : 'Contract Missing',
    'suspicious_title'     : 'Suspicious Title',
    'title_has_caps'       : 'Title ALL CAPS',
    'wide_salary_range'    : 'Wide Salary Range',
}

# Plot 1: Feature flag counts
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Feature Engineering — Fraud Indicator Distributions',
             fontsize=13, fontweight='bold')

# Group into 6 subplots - 2 features per bar group
feat_items = list(ALL_FEATURES.items())
for idx, ax in enumerate(axes.flat):
    i = idx * 2
    if i >= len(feat_items):
        ax.axis('off')
        continue
    group = feat_items[i:i+2]
    labels = [v for _, v in group]
    flagged = [df[k].sum() for k, _ in group]
    not_flagged = [len(df) - f for f in flagged]
    x = np.arange(len(group))
    bars1 = ax.bar(x - 0.2, not_flagged, 0.35, label='Not Flagged', color='#4CAF50')
    bars2 = ax.bar(x + 0.2, flagged,     0.35, label='Flagged',     color='#F44336')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    for b, v in zip(bars2, flagged):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+5,
                str(v), ha='center', fontsize=8, color='#F44336', fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS_PATH / '2_feature_flags.png', bbox_inches='tight')
plt.close()
print("Saved → 2_feature_flags.png")


Generating plots...
Saved → 2_feature_flags.png


In [19]:
# Plot 2: Risk score distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Fraud Risk Score Distribution', fontsize=13, fontweight='bold')

ax = axes[0]
ax.hist(df['fraud_risk_pct'], bins=25, color='steelblue', edgecolor='white')
ax.axvline(15, color='green',  linestyle='--', label='Low/Medium (15)')
ax.axvline(40, color='orange', linestyle='--', label='Medium/High (40)')
ax.set_title('Risk Score Distribution (0–100)', fontweight='bold')
ax.set_xlabel('Risk Score %')
ax.set_ylabel('Number of Jobs')
ax.legend()

ax = axes[1]
tc = df['risk_tier'].value_counts()
colors_t = {'Low': '#4CAF50', 'Medium': '#FF9800', 'High': '#F44336'}
ax.pie(tc.values, labels=tc.index,
       colors=[colors_t[t] for t in tc.index],
       autopct='%1.1f%%', startangle=90)
ax.set_title('Risk Tier Breakdown', fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS_PATH / '2_risk_distribution.png', bbox_inches='tight')
plt.close()
print("Saved → 2_risk_distribution.png")

Saved → 2_risk_distribution.png


In [20]:
# Plot 3: Correlation heatmap of all features
fig, ax = plt.subplots(figsize=(11, 9))
feat_cols = list(ALL_FEATURES.keys())
corr = df[feat_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5,
            xticklabels=list(ALL_FEATURES.values()),
            yticklabels=list(ALL_FEATURES.values()))
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_PATH / '2_correlation_heatmap.png', bbox_inches='tight')
plt.close()
print("Saved → 2_correlation_heatmap.png")

Saved → 2_correlation_heatmap.png


In [21]:
# FEATURE SUMMARY TABLE
print("\n--- Feature Summary ---")
summary = pd.DataFrame({
    'Feature'      : list(ALL_FEATURES.values()),
    'Column'       : list(ALL_FEATURES.keys()),
    'Flagged Count': [df[k].sum() for k in ALL_FEATURES.keys()],
    'Flagged %'    : [(df[k].mean()*100).round(1) for k in ALL_FEATURES.keys()],
    'Weight'       : [WEIGHTS[k] for k in ALL_FEATURES.keys()],
})
print(summary.to_string(index=False))


--- Feature Summary ---
                 Feature                Column  Flagged Count  Flagged %  Weight
         Missing Company       missing_company             20        0.7       3
             High Salary      high_salary_flag            123        4.5       2
          Salary Missing        salary_missing            276       10.1       1
       Short Description            short_desc              7        0.3       2
   Low Lexical Diversity low_lexical_diversity            137        5.0       2
     Suspicious Keywords   suspicious_keywords              2        0.1       3
Informal Contact in Desc       contact_in_desc              8        0.3       3
          Vague Location        vague_location            378       13.9       1
        Contract Missing      contract_missing            757       27.8       1
        Suspicious Title      suspicious_title             37        1.4       1
          Title ALL CAPS        title_has_caps            248        9.1       1
   